# 📊 تحلیل قیمت پژو ۲۰۷ دست دوم — دیوار

**دانشجو:** نام و نام خانوادگی  
**تاریخ:** ---

## داستان پروژه
می‌خواهم یک پژو ۲۰۷ بخرم. بین مدل‌های ۱۴۰۲ تا ۱۴۰۴ شک دارم و نمی‌دانم افت قیمت سالانه چقدر است. از طرفی شنیده‌ام ماشین در مشهد ارزان‌تر از تهران است. تصمیم گرفتم به جای حدس و گمان، داده‌های دیوار را تحلیل کنم.

**سوالات:**
1. توزیع کلی قیمت‌ها چگونه است؟
2. افت قیمت سالانه پژو ۲۰۷ چقدر است؟
3. آیا واقعاً ماشین در مشهد ارزان‌تر است؟
4. رابطه کارکرد و قیمت چیست؟

> ⚙️ **مرحله استخراج داده** جداگانه و به‌صورت Local انجام شده است.  
> فایل `scrape_divar.py` را روی سیستم خود اجرا کنید تا فایل `divar_listings.csv` تولید شود،  
> سپس آن را در Google Drive آپلود کرده و این نوت‌بوک را در Colab باز کنید.

---

## ۱. راه‌اندازی و بارگذاری داده‌ها

### ۱.۱ اتصال به Google Drive
فایل CSV را در Google Drive آپلود کرده، سپس سلول زیر را اجرا کنید تا Drive به Colab متصل شود.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### ۱.۲ import کتابخانه‌ها

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

### ۱.۳ بارگذاری فایل CSV
مسیر فایل را طبق محل آپلود در Drive خود تنظیم کنید.

In [ ]:
# ─── مسیر فایل خود را اینجا وارد کنید ───────────────────────────
FILE_PATH = '/content/drive/MyDrive/divar_listings.csv'
# ─────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(FILE_PATH, encoding='utf-8-sig')

print(f'ابعاد دیتاست: {df_raw.shape[0]} سطر × {df_raw.shape[1]} ستون')
print(f'ستون‌ها: {list(df_raw.columns)}')
df_raw.head()

---
## ۲. پاک‌سازی داده‌ها (Data Cleaning)

داده‌های دنیای واقعی شامل مقادیر گم‌شده، آگهی‌های توافقی و داده‌های پرت هستند.  
در این بخش گام‌به‌گام همه موارد را اصلاح می‌کنیم.

### ۲.۱ بررسی اولیه کیفیت داده‌ها

In [ ]:
print('─── نوع داده‌ها ─────────────────────')
print(df_raw.dtypes)
print()
print('─── مقادیر گم‌شده ──────────────────')
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(1)
print(pd.DataFrame({'تعداد': null_counts, 'درصد': null_pct}))
print()
print('─── نمونه ستون price_raw ───────────')
print(df_raw['price_raw'].value_counts().head(10))

### ۲.۲ حذف آگهی‌های توافقی و مقادیر خالی
آگهی‌هایی که قیمت، سال یا کارکرد ندارند قابل تحلیل نیستند.

In [ ]:
df = df_raw.copy()

before = len(df)

# حذف آگهی‌های بدون قیمت عددی (توافقی و خالی)
df = df.dropna(subset=['price_toman', 'year', 'mileage_km'])

# اطمینان از نوع عددی
df['price_toman'] = pd.to_numeric(df['price_toman'], errors='coerce')
df['year']        = pd.to_numeric(df['year'],        errors='coerce').astype('Int64')
df['mileage_km']  = pd.to_numeric(df['mileage_km'],  errors='coerce')

df = df.dropna(subset=['price_toman', 'year', 'mileage_km'])

print(f'سطرهای حذف شده (مقادیر خالی/توافقی): {before - len(df)}')
print(f'سطرهای باقی‌مانده: {len(df)}')

### ۲.۳ حذف داده‌های پرت (Outliers)
آگهی‌هایی با قیمت غیرواقعی (مثلاً ۱ تومان یا چند برابر قیمت بازار) را حذف می‌کنیم.  
از روش Percentile استفاده می‌کنیم: آگهی‌های زیر ۵٪ و بالای ۹۵٪ حذف می‌شوند.

In [ ]:
before = len(df)

# مرزهای قیمت
p05 = df['price_toman'].quantile(0.05)
p95 = df['price_toman'].quantile(0.95)

# مرزهای کارکرد (حذف کارکردهای غیرمنطقی)
km_max = 500_000

df_clean = df[
    (df['price_toman'] >= p05) &
    (df['price_toman'] <= p95) &
    (df['mileage_km']  <= km_max)
].copy()

print(f'محدوده قیمت مجاز: {p05/1e6:.0f}M تا {p95/1e6:.0f}M تومان')
print(f'سطرهای حذف شده (پرت): {before - len(df_clean)}')
print(f'دیتاست نهایی: {len(df_clean)} سطر')
df_clean.describe()

### ۲.۴ خلاصه داده‌های پاک‌شده

In [ ]:
print('─── توزیع آگهی بر اساس شهر ─────────')
print(df_clean['city'].value_counts())
print()
print('─── توزیع آگهی بر اساس مدل سال ─────')
print(df_clean['year'].value_counts().sort_index())

---
## ۳. تحلیل اکتشافی (EDA)

حالا با داده‌های پاک، به سوالات اولیه پاسخ می‌دهیم.

### سوال ۱ — توزیع کلی قیمت‌ها چگونه است؟
قبل از هر تحلیلی، باید بدانیم داده‌ها چه شکلی دارند: آیا توزیع نرمال است؟ دنباله‌دار است؟

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# هیستوگرام
sns.histplot(df_clean['price_toman'] / 1e6, bins=35, kde=True,
             color='steelblue', ax=axes[0])
axes[0].set_title('توزیع قیمت پژو ۲۰۷ دست دوم', fontsize=13)
axes[0].set_xlabel('قیمت (میلیون تومان)')
axes[0].set_ylabel('تعداد آگهی')
median_price = df_clean['price_toman'].median() / 1e6
axes[0].axvline(median_price, color='crimson', linestyle='--', linewidth=1.5,
                label=f'میانه: {median_price:.0f}M')
axes[0].legend()

# Boxplot
sns.boxplot(x=df_clean['price_toman'] / 1e6, color='steelblue', ax=axes[1])
axes[1].set_title('نمودار جعبه‌ای قیمت', fontsize=13)
axes[1].set_xlabel('قیمت (میلیون تومان)')

plt.tight_layout()
plt.savefig('charts/price_distribution.png', bbox_inches='tight')
plt.show()

print(f'میانگین: {df_clean["price_toman"].mean()/1e6:.1f}M')
print(f'میانه:   {df_clean["price_toman"].median()/1e6:.1f}M')
print(f'انحراف معیار: {df_clean["price_toman"].std()/1e6:.1f}M')

### سوال ۲ — افت قیمت سالانه چقدر است؟
میانه قیمت را بر اساس مدل سال مقایسه می‌کنیم تا ببینیم هر سال چقدر از ارزش ماشین کم می‌شود.

In [ ]:
# آمار هر مدل سال
year_stats = (
    df_clean.groupby('year')['price_toman']
    .agg(median='median', mean='mean', count='count', std='std')
    .reset_index()
)
year_stats['median_M'] = year_stats['median'] / 1e6

# افت قیمت نسبت به مدل قبل
year_stats = year_stats.sort_values('year')
year_stats['drop_pct'] = year_stats['median'].pct_change() * 100

print(year_stats[['year','median_M','count','drop_pct']].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(year_stats['year'].astype(str), year_stats['median_M'],
              color='steelblue', edgecolor='white', width=0.55)

# برچسب روی هر ستون
for bar, val in zip(bars, year_stats['median_M']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('میانه قیمت پژو ۲۰۷ بر اساس مدل سال', fontsize=13, pad=12)
ax.set_xlabel('مدل سال (شمسی)')
ax.set_ylabel('قیمت میانه (میلیون تومان)')
ax.set_ylim(0, year_stats['median_M'].max() * 1.18)

plt.tight_layout()
plt.savefig('charts/price_by_year.png', bbox_inches='tight')
plt.show()

### سوال ۳ — آیا ماشین در مشهد ارزان‌تر از تهران است؟
توزیع قیمت را بین شهرها مقایسه می‌کنیم.

In [ ]:
city_stats = (
    df_clean.groupby('city')['price_toman']
    .agg(median='median', count='count')
    .sort_values('median')
    .reset_index()
)
city_stats['median_M'] = city_stats['median'] / 1e6
print(city_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot مقایسه‌ای
city_order = city_stats['city'].tolist()
sns.boxplot(data=df_clean, x='city', y='price_toman',
            order=city_order, palette='Blues', ax=axes[0])
axes[0].set_title('توزیع قیمت به تفکیک شهر', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('قیمت (تومان)')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# Bar میانه
axes[1].bar(city_stats['city'], city_stats['median_M'],
            color='steelblue', edgecolor='white', width=0.5)
for i, (city, val) in enumerate(zip(city_stats['city'], city_stats['median_M'])):
    axes[1].text(i, val + 0.5, f'{val:.0f}M', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('میانه قیمت به تفکیک شهر', fontsize=13)
axes[1].set_ylabel('قیمت میانه (میلیون تومان)')
axes[1].set_ylim(0, city_stats['median_M'].max() * 1.2)

plt.tight_layout()
plt.savefig('charts/price_by_city.png', bbox_inches='tight')
plt.show()

# محاسبه درصد تفاوت
if 'تهران' in city_stats['city'].values and 'مشهد' in city_stats['city'].values:
    tehran  = city_stats.loc[city_stats['city']=='تهران',  'median'].values[0]
    mashhad = city_stats.loc[city_stats['city']=='مشهد', 'median'].values[0]
    diff    = (tehran - mashhad) / tehran * 100
    print(f'\nقیمت میانه تهران:  {tehran/1e6:.1f}M تومان')
    print(f'قیمت میانه مشهد: {mashhad/1e6:.1f}M تومان')
    print(f'تفاوت: {diff:.1f}٪ — {"مشهد ارزان‌تر است" if diff > 0 else "تهران ارزان‌تر است"}')

### سوال ۴ — رابطه کارکرد و قیمت چیست؟
انتظار داریم با افزایش کارکرد، قیمت کاهش یابد. نمودار پراکندگی و ضریب همبستگی این رابطه را نشان می‌دهد.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# رنگ‌بندی بر اساس شهر
cities  = df_clean['city'].unique()
palette = sns.color_palette('Set2', len(cities))

for city, color in zip(cities, palette):
    subset = df_clean[df_clean['city'] == city]
    ax.scatter(subset['mileage_km'] / 1000, subset['price_toman'] / 1e6,
               alpha=0.35, s=25, color=color, label=city)

# خط روند کلی
z = np.polyfit(df_clean['mileage_km'], df_clean['price_toman'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_clean['mileage_km'].min(), df_clean['mileage_km'].max(), 200)
ax.plot(x_line / 1000, p(x_line) / 1e6, 'r--', linewidth=2, label='خط روند کلی')

corr = df_clean['mileage_km'].corr(df_clean['price_toman'])

ax.set_title(f'رابطه کارکرد و قیمت  |  r = {corr:.2f}', fontsize=13, pad=12)
ax.set_xlabel('کارکرد (هزار کیلومتر)')
ax.set_ylabel('قیمت (میلیون تومان)')
ax.legend()

plt.tight_layout()
plt.savefig('charts/mileage_vs_price.png', bbox_inches='tight')
plt.show()

print(f'ضریب همبستگی پیرسون: {corr:.3f}')
print(f'تفسیر: {"همبستگی منفی قوی" if corr < -0.5 else "همبستگی منفی متوسط" if corr < -0.3 else "همبستگی ضعیف"}')

---
## ۴. نتیجه‌گیری

### پاسخ به سوالات اولیه

**سوال ۱ — توزیع قیمت:**  
> ✍️ اینجا بنویسید: آیا توزیع نرمال است؟ میانه و میانگین چقدر فاصله دارند؟

**سوال ۲ — افت قیمت سالانه:**  
> ✍️ اینجا بنویسید: میانه قیمت هر مدل سال چقدر است؟ افت سالانه چند درصد؟

**سوال ۳ — مقایسه شهری:**  
> ✍️ اینجا بنویسید: آیا مشهد ارزان‌تر بود؟ تفاوت چند درصد؟

**سوال ۴ — کارکرد و قیمت:**  
> ✍️ اینجا ضریب همبستگی و تفسیر آن را بنویسید.

### محدودیت‌های تحلیل
- داده‌ها در یک بازه زمانی کوتاه جمع‌آوری شده‌اند (قیمت بازار نوسان دارد)
- آگهی‌های توافقی از تحلیل حذف شدند
- عوامل دیگر (رنگ، تعویضی بودن قطعات) در داده نیستند